## Initialize

In [ ]:
import os
from datetime import datetime

print("=== BRONZE PHASE STARTED ===")
print(f"Start time: {datetime.now()}")

# Direct Volume path
csv_path = "/Volumes/data_engineering_workshop/kaggle/kaggle_api/creditcard.csv"

print("CSV Path:", csv_path)
print("File exists:", os.path.exists(csv_path))

# Debug
print("Files in folder:", os.listdir("/Volumes/data_engineering_workshop/kaggle/kaggle_api"))

## Initialize Spark & Define Bronze Schema

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (
    StructType, StructField, IntegerType, FloatType, 
    DoubleType, TimestampType, StringType
)

# Initialize Spark
spark = SparkSession.builder \
    .appName("bronze-creditcard") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

print("✓ Spark session initialized")

# Define Bronze Schema
BRONZE_SCHEMA = StructType([
    StructField("time", IntegerType(), False),
    StructField("v1", DoubleType(), False),
    StructField("v2", DoubleType(), False),
    StructField("v3", DoubleType(), False),
    StructField("v4", DoubleType(), False),
    StructField("v5", DoubleType(), False),
    StructField("v6", DoubleType(), False),
    StructField("v7", DoubleType(), False),
    StructField("v8", DoubleType(), False),
    StructField("v9", DoubleType(), False),
    StructField("v10", DoubleType(), False),
    StructField("v11", DoubleType(), False),
    StructField("v12", DoubleType(), False),
    StructField("v13", DoubleType(), False),
    StructField("v14", DoubleType(), False),
    StructField("v15", DoubleType(), False),
    StructField("v16", DoubleType(), False),
    StructField("v17", DoubleType(), False),
    StructField("v18", DoubleType(), False),
    StructField("v19", DoubleType(), False),
    StructField("v20", DoubleType(), False),
    StructField("v21", DoubleType(), False),
    StructField("v22", DoubleType(), False),
    StructField("v23", DoubleType(), False),
    StructField("v24", DoubleType(), False),
    StructField("v25", DoubleType(), False),
    StructField("v26", DoubleType(), False),
    StructField("v27", DoubleType(), False),
    StructField("v28", DoubleType(), False),
    StructField("amount", DoubleType(), False),
    StructField("class", IntegerType(), False),
    StructField("load_timestamp", TimestampType(), False),
    StructField("source_file", StringType(), False),
])

print(" Bronze schema defined")

## Load & Validate Data

In [ ]:
from pyspark.sql import functions as F

print("\n=== LOADING CSV WITH SPARK ===")

# --------------------------------------------
# STEP 1: Read CSV using Spark
# --------------------------------------------
sdf = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(csv_path)

# Normalize column names
sdf = sdf.toDF(*[col.strip().lower() for col in sdf.columns])

print(f"Initial rows: {sdf.count():,}")
print(f"Columns: {sdf.columns}")

# --------------------------------------------
# STEP 2: Validation
# --------------------------------------------
print("\n=== VALIDATING DATA ===")

sdf = sdf.filter(F.col("amount") >= 0)
sdf = sdf.filter(F.col("class").isin(0, 1))

print("✓ Validation complete")

# --------------------------------------------
# STEP 3: Add metadata
# --------------------------------------------
sdf = sdf.withColumn("load_timestamp", F.current_timestamp()) \
         .withColumn("source_file", F.lit("creditcard.csv"))

print("✓ Metadata added")

In [ ]:
print(f"\n=== CREATING DELTA TABLE ===")

# --------------------------------------------
# STEP: Define table name (bronze layer)
# --------------------------------------------
table_name = "data_engineering_workshop.creditcard.creditcard_bronze"

print(f"Writing to: {table_name}")

# --------------------------------------------
# STEP: Write to Unity Catalog
# --------------------------------------------
sdf.write \
    .mode("overwrite") \
    .format("delta") \
    .option("mergeSchema", "true") \
    .saveAsTable(table_name)

print(" Bronze table created successfully!")

# --------------------------------------------
# STEP: Verification
# --------------------------------------------
verify = spark.read.table(table_name)

print(f"\nVerification: {verify.count():,} rows in {table_name}")
verify.show(5)